# Getting Sonde Data Directly from the Integrated Global Radiosonde Archive (IGRA) 

At the moment, MELODIES-MONET can directly take an upper air data file downloaded from the University of Wyoming (https://weather.uwyo.edu/upperair/sounding.shtml). Users may prefer to grab their upper air data from the Integrated Global Radiosonde Archive (IGRA) (https://www.ncei.noaa.gov/products/weather-balloon/integrated-global-radiosonde-archive). The Jupyter notebook below will show you how to grab a site from the IGRA and save its data as a .csv. NOTE: The primary disadvantage to using the IGRA is the lack of time varying latitude and longitude.

To find specific sites, users will want to refer to this list: https://www.ncei.noaa.gov/data/integrated-global-radiosonde-archive/doc/igra2-station-list.txt

Additional documentation on the dataset can be found here: https://www.ncei.noaa.gov/data/integrated-global-radiosonde-archive/doc/igra2-data-format.txt

MELODIES-MONET capabilities with IGRA datasets continue to be developed. 

In [4]:
# import libraries
from datetime import datetime, timedelta
import pandas as pd

# siphon may need to be optional dep. 
from siphon.simplewebservice.igra2 import IGRAUpperAir

# provide a date / time to pull sounding from 
# year, month, day, hour (UTC)
date = datetime(2014, 9, 10, 0)

# station number from NCEI.txt file 
# Users can look up specific sites via this link:
# https://www.ncei.noaa.gov/data/integrated-global-radiosonde-archive/doc/igra2-station-list.txt
station = 'USM00070026'
df, header = IGRAUpperAir.request_data(date, station)

df['latitude'] = header['latitude']
df['longitude'] = header['longitude']

# convert the elapsed time to seconds
def etime_to_seconds(etime):
    try:
        etime = int(etime)
        if etime in (-8888, -9999):
            return float('nan')
        minutes = etime // 100
        seconds = etime % 100
        return minutes * 60 + seconds
    except Exception:
        return float('nan')

df['etime'] = pd.to_numeric(df['etime'], errors='coerce')
df['etime_seconds'] = df['etime'].apply(etime_to_seconds)

# Parse header values
def get_scalar(val):
    if isinstance(val, pd.Series):
        return val.iloc[0]
    elif isinstance(val, pd.DataFrame):
        return val.iloc[0, 0]
    return val

date_val = get_scalar(header['date'])
if not isinstance(date_val, str):
    date_val = str(date_val)
date_dt = pd.to_datetime(date_val)

hour_val = get_scalar(header['hour'])
if not isinstance(hour_val, int):
    hour_val = int(hour_val)

release_time_val = get_scalar(header.get('release_time', None))

# Build launch time safely
if release_time_val not in (None, '', 0):
    # IGRA release_time is MMMM (minutes since midnight)
    rt = int(str(release_time_val).zfill(4))
    launch_datetime = date_dt + timedelta(minutes=rt//100, seconds=rt%100)
else:
    launch_datetime = date_dt + timedelta(hours=hour_val)

# add the column 
df['time'] = launch_datetime + pd.to_timedelta(df['etime_seconds'], unit='s')

# debugging 
#print(df[['etime', 'etime_seconds', 'time']].head())

# drop unnecessary columns
#df.columns
df.drop("etime", axis = 1, inplace=True) # pretty sure u_wind and v_wind can be dropped. but we can keep them for extra eval if we like. 

# save output to .csv.
# From here, users can take the file and pull it into rest of melodies-monet
df.to_csv('sounding_data.csv', index=False)

In [5]:
# example dataframe
df

,lvltyp1,lvltyp2,pressure,pflag,height,zflag,temperature,tflag,relative_humidity,direction,speed,date,u_wind,v_wind,dewpoint,latitude,longitude,etime_seconds,time
0,2,1,1020.95,2,15,0,1.7,2,82.0,57,7.2,2014-09-10,-6.0,-3.9,-1.0,71.2889,-156.7833,0,2014-09-10 01:30:00
1,2,0,1018.16,2,37,2,1.4,2,75.1,64,7.0,2014-09-10,-6.3,-3.1,-2.5,NaN,NaN,5,2014-09-10 01:30:05
2,2,0,1003.21,0,156,2,0.0,2,81.4,89,7.5,2014-09-10,-7.5,-0.1,-2.8,NaN,NaN,32,2014-09-10 01:30:32
3,1,0,1000.00,0,182,2,-0.3,2,82.8,88,7.7,2014-09-10,-7.7,-0.3,-2.9,NaN,NaN,38,2014-09-10 01:30:38
4,2,0,983.95,0,311,2,-1.7,2,89.9,78,9.2,2014-09-10,-9.0,-1.9,-3.1,NaN,NaN,69,2014-09-10 01:31:09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226,3,0,NaN,0,32197,0,NaN,0,NaN,258,13.3,2014-09-10,13.0,2.8,NaN,NaN,NaN,3920,2014-09-10 02:35:20
227,3,0,NaN,0,32549,0,NaN,0,NaN,275,10.7,2014-09-10,10.7,-0.9,NaN,NaN,NaN,3940,2014-09-10 02:35:40
228,3,0,NaN,0,32894,0,NaN,0,NaN,283,11.6,2014-09-10,11.3,-2.6,NaN,NaN,NaN,3960,2014-09-10 02:36:00
229,3,0,NaN,0,33236,0,NaN,0,NaN,280,8.6,2014-09-10,8.5,-1.5,NaN,NaN,NaN,4020,2014-09-10 02:37:00
